## Sequence-Based Recommender (LSTM)

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

os.chdir(r"C:\Users\Saeed\Desktop\neural_network_project")

PROCESSED_DIR = "data/processed"
MODELS_DIR    = "models/sequence"
os.makedirs(MODELS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("Ready!")

Device: cpu
Ready!


In [2]:
print("Loading data...")
ratings = pd.read_csv(f"{PROCESSED_DIR}/collaborative_ratings.csv")
master  = pd.read_csv(f"{PROCESSED_DIR}/movies_master.csv")

print(f"Ratings : {len(ratings):,}")
print(f"Users   : {ratings['userId'].nunique():,}")
print(f"Movies  : {ratings['movie_id'].nunique():,}")
ratings.head()

Loading data...
Ratings : 21,451,796
Users   : 156,066
Movies  : 7,292


,userId,movie_id,rating
0,1,680,5.0
1,3,680,5.0
2,4,680,4.0
3,5,680,4.0
4,7,680,4.0


In [3]:
print("Building user sequences...")

user_counts  = ratings["userId"].value_counts()
active_users = user_counts[user_counts >= 10].index
ratings_f    = ratings[ratings["userId"].isin(active_users)].copy()
ratings_f    = ratings_f.sort_values(["userId","rating"], ascending=[True,False])

# ── خد sample بس 20,000 user عشوائي ──────────────────────
sampled_users = np.random.choice(active_users, size=min(20000, len(active_users)), replace=False)
ratings_f     = ratings_f[ratings_f["userId"].isin(sampled_users)]

all_movies  = ratings_f["movie_id"].unique()
movie2id    = {m: i+1 for i, m in enumerate(all_movies)}
id2movie    = {i: m for m, i in movie2id.items()}
n_movies    = len(movie2id) + 1

ratings_f["movie_enc"] = ratings_f["movie_id"].map(movie2id)

user_sequences = {}
for uid, group in ratings_f.groupby("userId"):
    seq = group["movie_enc"].tolist()
    if len(seq) >= 5:
        user_sequences[uid] = seq

print(f"Users            : {len(user_sequences):,}")
print(f"Vocabulary size  : {n_movies:,}")
print(f"Avg seq length   : {np.mean([len(s) for s in user_sequences.values()]):.1f}")

Building user sequences...
Users            : 20,000
Vocabulary size  : 6,933
Avg seq length   : 136.7


In [4]:
class SequenceDataset(Dataset):
    def __init__(self, sequences, max_len=20, max_samples=500000):
        self.samples = []
        self.max_len = max_len
        
        for uid, seq in sequences.items():
            for i in range(2, len(seq) + 1):
                input_seq = seq[max(0, i - max_len - 1): i - 1]
                target    = seq[i - 1]
                self.samples.append((input_seq, target))
                
                # وقف لما نوصل للحد
                if len(self.samples) >= max_samples:
                    break
            if len(self.samples) >= max_samples:
                break
        
        # shuffle
        import random
        random.shuffle(self.samples)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        input_seq, target = self.samples[idx]
        padded = [0] * (self.max_len - len(input_seq)) + input_seq
        return (
            torch.tensor(padded, dtype=torch.long),
            torch.tensor(target, dtype=torch.long)
        )

MAX_LEN    = 20
BATCH_SIZE = 1024   # زودنا الـ batch عشان يخلص أسرع

dataset    = SequenceDataset(user_sequences, max_len=MAX_LEN, max_samples=500000)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f"Total samples : {len(dataset):,}")
print(f"Batches       : {len(dataloader):,}")

Total samples : 500,000
Batches       : 489


In [5]:
class MovieLSTM(nn.Module):
    """
    LSTM لتوقع الفيلم التالي في تسلسل المشاهدة
    """
    def __init__(self, n_movies, embed_dim=64, hidden_dim=128, n_layers=2, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Embedding(
            num_embeddings = n_movies,
            embedding_dim  = embed_dim,
            padding_idx    = 0
        )
        
        self.lstm = nn.LSTM(
            input_size  = embed_dim,
            hidden_size = hidden_dim,
            num_layers  = n_layers,
            batch_first = True,
            dropout     = dropout
        )
        
        self.dropout = nn.Dropout(dropout)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, n_movies)
        )
    
    def forward(self, x):
        # x: (batch, seq_len)
        emb  = self.dropout(self.embedding(x))   # (batch, seq_len, embed_dim)
        out, _ = self.lstm(emb)                   # (batch, seq_len, hidden_dim)
        last = out[:, -1, :]                      # آخر hidden state
        return self.fc(last)                      # (batch, n_movies)


model_lstm = MovieLSTM(
    n_movies  = n_movies,
    embed_dim = 64,
    hidden_dim= 128,
    n_layers  = 2,
    dropout   = 0.3
).to(device)

# حساب عدد الـ parameters
total_params = sum(p.numel() for p in model_lstm.parameters())
print(f"Model parameters: {total_params:,}")
print(model_lstm)

Model parameters: 2,489,941
MovieLSTM(
  (embedding): Embedding(6933, 64, padding_idx=0)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=6933, bias=True)
  )
)


In [6]:
EPOCHS    = 10
LR        = 0.001

optimizer = torch.optim.Adam(model_lstm.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print(f"Training for {EPOCHS} epochs...")
print(f"Dataset: {len(dataset):,} samples | Batch: {BATCH_SIZE}")
print("-" * 50)

history = []

for epoch in range(1, EPOCHS + 1):
    model_lstm.train()
    total_loss = 0
    correct    = 0
    total      = 0
    
    for batch_x, batch_y in dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        output = model_lstm(batch_x)        # (batch, n_movies)
        loss   = criterion(output, batch_y)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_lstm.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        preds       = output.argmax(dim=1)
        correct    += (preds == batch_y).sum().item()
        total      += len(batch_y)
    
    scheduler.step()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total * 100
    history.append({"epoch": epoch, "loss": avg_loss, "accuracy": accuracy})
    
    print(f"Epoch {epoch:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Acc: {accuracy:.2f}% | LR: {scheduler.get_last_lr()[0]:.6f}")

print("\nTraining complete!")

Training for 10 epochs...
Dataset: 500,000 samples | Batch: 1024
--------------------------------------------------
Epoch  1/10 | Loss: 7.2174 | Acc: 0.75% | LR: 0.001000
Epoch  2/10 | Loss: 6.0243 | Acc: 2.55% | LR: 0.001000
Epoch  3/10 | Loss: 5.5851 | Acc: 3.78% | LR: 0.000500
Epoch  4/10 | Loss: 5.3511 | Acc: 4.61% | LR: 0.000500
Epoch  5/10 | Loss: 5.2461 | Acc: 5.05% | LR: 0.000500
Epoch  6/10 | Loss: 5.1623 | Acc: 5.44% | LR: 0.000250
Epoch  7/10 | Loss: 5.0879 | Acc: 5.83% | LR: 0.000250
Epoch  8/10 | Loss: 5.0508 | Acc: 6.03% | LR: 0.000250
Epoch  9/10 | Loss: 5.0193 | Acc: 6.19% | LR: 0.000125
Epoch 10/10 | Loss: 4.9870 | Acc: 6.32% | LR: 0.000125

Training complete!


In [7]:
model_lstm.eval()

def predict_next_movies(watched_movie_ids: list, n: int = 10) -> pd.DataFrame:
    """
    توقع الأفلام التالية بناءً على قائمة أفلام شافها المستخدم
    watched_movie_ids: list of TMDB movie_ids
    """
    # تحويل لـ encoded IDs
    encoded = [movie2id[mid] for mid in watched_movie_ids if mid in movie2id]
    
    if not encoded:
        print("No movies found in training data!")
        return pd.DataFrame()
    
    # Padding
    padded = [0] * max(0, MAX_LEN - len(encoded)) + encoded[-MAX_LEN:]
    x = torch.tensor([padded], dtype=torch.long).to(device)
    
    with torch.no_grad():
        logits = model_lstm(x)                    # (1, n_movies)
        probs  = torch.softmax(logits[0], dim=0)  # احتمالات كل فيلم
    
    # Top-N بعد استبعاد الأفلام المشاهدة
    probs_np = probs.cpu().numpy()
    probs_np[encoded] = 0  # شيل الأفلام اللي شافها
    probs_np[0] = 0        # شيل الـ padding
    
    top_encoded = np.argsort(probs_np)[::-1][:n*3]
    
    results = []
    for enc_id in top_encoded:
        if enc_id not in id2movie:
            continue
        mid = id2movie[enc_id]
        row = master[master["movie_id"] == mid]
        if row.empty:
            continue
        results.append({
            "movie_id"    : mid,
            "title"       : row.iloc[0]["title"],
            "genres"      : row.iloc[0]["genres"],
            "vote_average": row.iloc[0]["vote_average"],
            "poster_path" : row.iloc[0]["poster_path"],
            "score"       : round(float(probs_np[enc_id]), 6)
        })
        if len(results) == n:
            break
    
    return pd.DataFrame(results)

print("predict_next_movies() ready!")

predict_next_movies() ready!


In [8]:
# اختار أفلام مشهورة كـ input
test_sequences = {
    "Nolan Fan": ["Inception", "The Dark Knight", "Interstellar"],
    "Pixar Fan": ["Toy Story", "Finding Nemo", "The Incredibles"],
    "Crime Fan": ["The Godfather", "Pulp Fiction", "Goodfellas"],
}

print("=== Sequence-Based Predictions ===\n")

for fan_type, movie_titles in test_sequences.items():
    # جيب الـ movie_ids
    ids = []
    for t in movie_titles:
        row = master[master["title"].str.lower() == t.lower()]
        if not row.empty:
            ids.append(row.iloc[0]["movie_id"])
    
    if not ids:
        continue
    
    print(f"[{fan_type}] Watched: {', '.join(movie_titles)}")
    recs = predict_next_movies(ids, n=5)
    
    if not recs.empty:
        for _, row in recs.iterrows():
            print(f"  → {row['title']:40s} score: {row['score']:.6f}")
    print()

=== Sequence-Based Predictions ===

[Nolan Fan] Watched: Inception, The Dark Knight, Interstellar
  → Whiplash                                 score: 0.049400
  → The Imitation Game                       score: 0.047111
  → Ex Machina                               score: 0.043406
  → The Martian                              score: 0.039262
  → Guardians of the Galaxy                  score: 0.032914

[Pixar Fan] Watched: Toy Story, Finding Nemo, The Incredibles
  → Fight Club                               score: 0.039893
  → The Silence of the Lambs                 score: 0.035993
  → Batman Begins                            score: 0.026990
  → Harry Potter and the Goblet of Fire      score: 0.025996
  → The Usual Suspects                       score: 0.025542

[Crime Fan] Watched: The Godfather, Pulp Fiction, Goodfellas
  → The Godfather Part II                    score: 0.088650
  → Fight Club                               score: 0.065703
  → Alien                                    

In [9]:
print("Saving model...")

torch.save(model_lstm.state_dict(), f"{MODELS_DIR}/lstm_model.pt")
joblib.dump(movie2id, f"{MODELS_DIR}/movie2id.pkl")
joblib.dump(id2movie, f"{MODELS_DIR}/id2movie.pkl")
joblib.dump({
    "n_movies"  : n_movies,
    "embed_dim" : 64,
    "hidden_dim": 128,
    "n_layers"  : 2,
    "max_len"   : MAX_LEN
}, f"{MODELS_DIR}/lstm_config.pkl")

# حفظ الـ history
pd.DataFrame(history).to_csv(f"{MODELS_DIR}/training_history.csv", index=False)

print("Saved:")
for f in os.listdir(MODELS_DIR):
    size = os.path.getsize(f"{MODELS_DIR}/{f}") / 1024 / 1024
    print(f"  {f:35s} {size:.2f} MB")

Saving model...
Saved:
  id2movie.pkl                        0.15 MB
  lstm_config.pkl                     0.00 MB
  lstm_model.pt                       9.50 MB
  movie2id.pkl                        0.15 MB
  training_history.csv                0.00 MB
